## 不使用@Tool装饰器

底层会调用convert_to_openai_tool函数转换为json_schema格式，注意各种说明要写清楚写规范。

### convert_to_openai_tool

执行model.bind_tools([get_weather])，底层最终会调用convert_to_openai_tool生成工具描述。所以可以直接调用后者查看解析后的工具描述。

**结果字段说明：**
- (1) `type`：定义当前数据节点必须是什么数据类型。常见类型有 `string, number, integer, boolean, object, array, null`。`object` 即是json对象。
- (2) `properties`：用于定义JSON对象（Object）中可以包含哪些属性（键），以及每个属性对应的值类型和说明。
- (3) `required`：当 `type` 为 `"object"` 时使用，是一个数组，列出了对象中必须存在的属性名。

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
# 从.env文件中加载环境变量
load_dotenv(override=True)
OPENROUTER_API_KEY = os.getenv("OPENROUTER_API_KEY")
OPENROUTER_BASE_URL = os.getenv("OPENROUTER_BASE_URL")
model = init_chat_model(
    model="gpt-5.4-mini",
    model_provider="openai",
    api_key=OPENROUTER_API_KEY,
    base_url=OPENROUTER_BASE_URL
)

In [1]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from rich import print as rich_print


def get_weather(city: str):
    return f"{city}天气晴朗"
rich_print(convert_to_openai_tool(get_weather))


{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## 工具各种说明

### description说明

convert_to_openai_tool会从docstring(文档字符串)加载工具的描述信息

### 参数说明

`convert_to_openai_tool` 会从 docstring 加载参数说明，这里的 docstring 必须遵循 Google 风格。
- Google 风格 docstring 说明：https://google.github.io/styleguide/pyguide.html
- Google 风格 docstring 示例：https://www.sphinx-doc.org/en/master/usage/extensions/example_google.html

使用Args:、Returns:、Raises:等关键字，这种方式可读性强。Agent通过工具的这些注释来理解工具的用途和调用时机，因此清晰、准确的文档字符串是工具能被正确调用的前提。

### 参数类型说明

工具函数的参数类型会在JSON_Schema中被描述。删除了参数类型注解，则工具描述中不包含参数类型说明

注意：如果docstring中包含参数说明，则对应的参数必须有类型注解，否则报错

###  参数默认值说明

如果参数没有默认值，则会包含在required对应的列表中。反之，则参数的描述信息会包含default字段，并且不会出现在required列表中。

In [4]:

def get_weather(date: str,city: str = "北京") -> str:
    """
    获取指定日期在指定城市的天气。

    Args:
        date: 日期，格式为YYYY-MM-DD。
        city: 城市名称，默认值为"北京"。
        
    Returns:
        天气描述，例如"晴朗"。
    """
    return f"{date} {city}天气晴朗。"
    
rich_print(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取指定日期在指定城市的天气。',
        'parameters': {
            'properties': {
                'date': {'description': '日期，格式为YYYY-MM-DD。', 'type': 'string'},
                'city': {'default': '北京', 'description': '城市名称，默认值为"北京"。', 'type': 'string'}
            },
            'required': ['date'],
            'type': 'object'
        }
    }
}

##  使用@tool装饰器

### 基本定义

和不使用没什么太大差别，主要是使用@tool装饰器有更多自由度，下面是标准定义，通常情况下推荐这样的写法。

In [6]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain_core.tools import tool

from rich import print as rprint


@tool(parse_docstring=True)
def get_weather(city: str, units: str = "celsius", include_forecast: bool = False) -> str:
    """
    获取当日天气，可选择是否同时查询未来五日天气预报

    Args:
        city: 城市
        units: 气温单位，可选：celsius-摄氏度，fahrenheit-华氏度
        include_forecast: 是否包含未来五日的天气预报
    
    Returns:
        具体的天气描述
    """
    temp = 22 if units == "celsius" else 72
    result = f'{city}当天气温：{temp} {"摄氏度" if units == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '获取当日天气，可选择是否同时查询未来五日天气预报',
        'parameters': {
            'properties': {
                'city': {'description': '城市', 'type': 'string'},
                'units': {
                    'default': 'celsius',
                    'description': '气温单位，可选：celsius-摄氏度，fahrenheit-华氏度',
                    'type': 'string'
                },
                'include_forecast': {
                    'default': False,
                    'description': '是否包含未来五日的天气预报',
                    'type': 'boolean'
                }
            },
            'required': ['city'],
            'type': 'object'
        }
    }
}

### name_or_callable与description参数

- name_or_callable运行自定义工具名称，但通常就使用函数名称，不推荐使用此参数

- description参数是工具的描述，优先级更高会覆盖docstring的描述



In [7]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
from rich import print as rprint
@tool(description="根据城市名称查询当日天气的工具")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'get_weather',
        'description': '根据城市名称查询当日天气的工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

In [9]:
from langchain_core.utils.function_calling import convert_to_openai_tool
from langchain.tools import tool
@tool(name_or_callable="getWeather")
def get_weather(city: str):
    """
    天气查询工具
    """
    return f"{city}天气晴朗"
rprint(convert_to_openai_tool(get_weather))

{
    'type': 'function',
    'function': {
        'name': 'getWeather',
        'description': '天气查询工具',
        'parameters': {'properties': {'city': {'type': 'string'}}, 'required': ['city'], 'type': 'object'}
    }
}

## args_schema工具参数定义

工具参数除了直接在函数签名中定义，还可以使用Pydantic模型和JSON Schema来定义参数。

- Pydantic模型：语义更明确，支持类型检查和默认值
- JSON Schema：更通用，支持自定义验证规则与动态拼接，工具参数模式可以基于数据库配置或用户输入在运行时动态生成，所以这种方式特别适合参数结
构需要动态生成的场景。


In [ ]:

from pydantic import BaseModel, Field
from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from typing import Literal


class WeatherInput(BaseModel):
    city: str = Field(
        default="北京",
        description="城市"
    )
    unit: Literal["celsius", "fahrenheit"] = Field(
        default="celsius",
        description="气温单位"
    )
    include_forecast: bool = Field(
        default=False,
        description="是否包含未来五日天气预报"
    )

@tool(args_schema=WeatherInput)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = False) -> str:
    """获取当日天气，可选未来五日天气预报"""
    temp = 22 if unit == "celsius" else 72
    result = f'{city}当天气温：{temp} {"摄氏度" if unit == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


convert_to_openai_tool(get_weather)

外层完整工具结构：
```json
{
  "type": "function",
  "function": {
    "name": "get_weather",
    "description": "获取当日天气，可选未来五日天气预报",
    "parameters": {
      "type": "object",
      "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
      },
      "required": ["location", "units", "include_forecast"]
    }
  }
}
```

说明：应该传递给`args_schema`的只有`parameters`对应的JSON字符串，即：
```json
{
  "type": "object",
  "properties": {
    "location": {"type": "string"},
    "units": {"type": "string"},
    "include_forecast": {"type": "boolean"}
  },
  "required": ["location", "units", "include_forecast"]
}
```

In [11]:

from langchain.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool

weather_schema = {
    "type": "object",
    "properties": {
        "location": {"type": "string"},
        "units": {"type": "string"},
        "include_forecast": {"type": "boolean"}
    },
    "required": ["location", "units", "include_forecast"]
}


@tool(args_schema=weather_schema)
def get_weather(city: str, unit: str = "celsius", include_forecast: bool = False) -> str:
    """获取当日天气，可选未来五日天气预报"""
    temp = 22 if unit == "celsius" else 72
    result = f'{city}当天气温：{temp} {"摄氏度" if unit == "celsius" else "华氏度"}'
    if include_forecast:
        result += "\n未来五天都是晴天"
    return result


print(convert_to_openai_tool(get_weather))


{'type': 'function', 'function': {'name': 'get_weather', 'description': '获取当日天气，可选未来五日天气预报', 'parameters': {'type': 'object', 'properties': {'location': {'type': 'string'}, 'units': {'type': 'string'}, 'include_forecast': {'type': 'boolean'}}, 'required': ['location', 'units', 'include_forecast']}}}
